# Exploratory Data Analysis — Heineken Demand Forecasting

## Business Context
The Demand Planning team needs to **predict product demand 8 weeks in advance** per SKU × supermarket combination.
A senior planner's retirement has led to increased stock-outs and write-offs, creating direct brand and financial risk.

This EDA answers two questions:
1. What is the quality of our data, and what cleaning is needed?
2. What patterns and features should drive our forecasting model?


## 1. Setup

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from utils import read_demand, read_promotions, extend_promotions_days, merge, clean_demand_per_group, aggregate_to_weekly

# Reproducibility
np.random.seed(42)


## 2. Load Data

Two datasets:
- **demand.csv** — daily demand per SKU × supermarket (2019–2021)
- **promotions.csv** — promotion start dates (each promotion lasts exactly 7 days)


In [ ]:
demand = read_demand("./demand.csv")
promotions = read_promotions("./promotions.csv")
print(f"Demand shape: {demand.shape}")
print(f"Promotions shape: {promotions.shape}")
demand.head()


In [ ]:
promotions.head()


## 3. Part 1: Data Quality

### 3.1 Basic Overview


In [ ]:
print("=== DEMAND OVERVIEW ===")
print(f"Date range: {demand.index.min().date()} to {demand.index.max().date()}")
print(f"SKUs: {demand.sku.unique().tolist()}")
print(f"Supermarkets: {demand.supermarket.unique().tolist()}")
print(f"Total rows: {len(demand)}")
print(f"\n=== MISSING VALUES ===")
print(f"Total NaN in demand: {demand.demand.isnull().sum()} ({demand.demand.isnull().mean()*100:.1f}% of rows)")
print(f"\n=== PER GROUP NULL COUNTS ===")
null_by_group = demand[demand.demand.isnull()].groupby(['sku', 'supermarket']).size()
print(null_by_group.to_string())


In [ ]:
# Check expected row count vs actual
expected_days = (demand.index.max() - demand.index.min()).days + 1
print(f"Expected days in range: {expected_days}")
print(f"Expected rows (3 SKU × 3 stores × {expected_days} days): {3 * 3 * expected_days}")
print(f"Actual rows: {len(demand)}")
print(f"Rows missing: {3 * 3 * expected_days - len(demand)}")


### 3.2 Outlier Detection

We use z-score to detect statistical outliers. A z-score > 3 flags values more than 3 standard deviations from the group mean.

**Why group-level stats?** Desperados in Jumbo has ~2.4× the mean demand of Heineken 0.0 in Dirk, so a single global z-score would flag normal high-demand weeks as outliers.


In [ ]:
# Outlier analysis per group
outlier_summary = []
for (sku, sm), grp in demand.groupby(['sku', 'supermarket']):
    z = (grp.demand - grp.demand.mean()) / grp.demand.std()
    n_outliers = (z.abs() > 3).sum()
    max_demand = grp.demand.max()
    mean_demand = grp.demand.mean()
    outlier_summary.append({
        'SKU': sku, 'Supermarket': sm,
        'Mean Demand': round(mean_demand, 1),
        'Max Demand': max_demand,
        'Outliers (z>3)': n_outliers,
        '% Outliers': round(n_outliers / len(grp) * 100, 1)
    })

pd.DataFrame(outlier_summary).set_index(['SKU','Supermarket'])


**Observation:** Outliers (~0.5–1%) are likely promotion periods. Rather than removing them, we should model them explicitly via the promotion feature.

### 3.3 Imputation Strategy

**Options considered:**

| Method | Pros | Cons | Decision |
|--------|------|------|----------|
| Drop NaN rows | Simple | Loses 11% of data; creates gaps in time series | ❌ |
| Forward fill | Preserves trends | Propagates stale values too far | ❌ |
| Backward fill | Good for gaps at start | Only works if next value exists | ✓ (primary) |
| Group mean | Always works | Loses temporal structure | ✓ (fallback) |
| Interpolation | Smooth transitions | Can smooth out real demand spikes | ❌ for production |

**Decision:** `bfill` within group, fallback to group mean. Imputing **per group** avoids cross-contamination across products/stores with different demand scales.


In [ ]:
cleaned_demand = clean_demand_per_group(demand.copy())
print(f"NaN before cleaning: {demand.demand.isnull().sum()}")
print(f"NaN after cleaning:  {cleaned_demand.demand.isnull().sum()}")


### 3.4 Merge with Promotions

The promotion data only has start dates. We extend each promotion to 7 days, then left-join on (date, sku, supermarket).


In [ ]:
extended_promotions = extend_promotions_days(promotions, 7).drop("promotion_id", axis=1)
df_daily = merge(cleaned_demand, extended_promotions)
print(f"Merged daily data shape: {df_daily.shape}")
print(f"Promotion days: {df_daily.promotion.sum()} ({df_daily.promotion.mean()*100:.1f}% of all days)")
df_daily.head()


## 4. Part 2: Exploration

### 4.1 Aggregate to Weekly

The business wants 8-week-ahead forecasts. Weekly aggregation: 
- Reduces daily noise
- Aligns with promotion duration (1 week = natural unit)
- Creates a cleaner signal-to-noise ratio


In [ ]:
weekly = aggregate_to_weekly(df_daily)
print(f"Weekly data shape: {weekly.shape}")
print(f"\nWeeks per SKU × Store:")
print(weekly.groupby(['sku','supermarket']).size().unstack())
weekly.head()


### 4.2 Demand Over Time

In [ ]:
from IPython.display import Image
Image('eda_fig2_timeseries.png')


In [ ]:
# Summary statistics per group
stats = weekly.groupby(['sku','supermarket'])['demand'].agg(['mean','std','min','max','count'])
stats.columns = ['Mean (weekly)', 'Std Dev', 'Min', 'Max', 'Weeks']
stats.round(1)


**Key observations:**
1. **Desperados** has the highest and most volatile demand (highest std/mean ratio → coefficient of variation ~0.29), making it the hardest to forecast
2. **Heineken Regular** is the most stable (CV ~0.13), easiest to forecast
3. All series span ~156 weeks with consistent coverage


### 4.3 Seasonality Analysis

In [ ]:
Image('eda_fig3_seasonality_promo.png')


In [ ]:
# Promotion lift quantification
print("=== PROMOTION LIFT ===")
for sku in weekly.sku.unique():
    grp = weekly[weekly.sku == sku]
    on = grp[grp.promotion==True]['demand'].mean()
    off = grp[grp.promotion==False]['demand'].mean()
    print(f"  {sku:20s}: {off:.0f} → {on:.0f} on promo (+{(on/off-1)*100:.1f}% lift)")


**Key findings:**

| Insight | Business implication |
|---------|---------------------|
| **Desperados** shows +14% demand lift during promotions | Must include future promotion calendar in forecast |
| **Heineken 0.0** shows +6% lift | Moderate promo effect |
| **Heineken Regular** shows +5% lift | Low promo sensitivity |
| Monthly demand is relatively flat (no strong summer/winter pattern) | Calendar features are less critical than in e.g. ice cream |
| Day-of-week effect is minimal (consistent across Mon–Sun) | Daily pattern is not a key driver at weekly aggregation |

### 4.4 Feature Candidates for Modelling

Based on the EDA, these features should enter the model:

1. **Lag features** at 8, 9, 10, 12, 13 weeks (minimum lag = 8 to respect forecast horizon)
2. **52-week lag** (same week last year — captures any annual pattern)
3. **Rolling mean / std** over 4, 8, 13 weeks (trend + volatility context)
4. **Promotion flag** (known 8 weeks ahead from planning calendar)
5. **Month / week-of-year** (seasonality)
6. **SKU and supermarket identity** (encoded)


## 5. Summary for the Business

| Question | Answer |
|----------|--------|
| How much data do we have? | 3 years (Jan 2019–Dec 2021), 9 SKU×store series, ~156 weeks each |
| Data quality issues? | ~11% missing daily values — imputable per group with bfill + mean fallback |
| Which product is hardest to forecast? | Desperados (highest CV = 0.29) |
| Does promotion affect demand? | Yes — up to +14% lift (Desperados). Promotions must be in the model. |
| Any strong seasonality? | Mild. Monthly demand is relatively flat, making lag-based features most powerful |
| Ready for modelling? | ✅ After cleaning + weekly aggregation + feature engineering |
